# SpaceX Falcon 9 Data Wrangling

This notebook prepares SpaceX Falcon 9 launch data for exploratory data analysis and predictive modeling.

The workflow includes loading the collected data, inspecting the dataset, handling missing values, selecting relevant variables, and preparing the final analytical dataset.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

## 2. Collect Launch Data

The SpaceX API is used to obtain the historical launch records required for the analysis.

In [ ]:
import requests

url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(url)
response.raise_for_status()

launches = response.json()
df = pd.json_normalize(launches)

print("Initial dataset shape:", df.shape)

## 3. Inspect the Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())

df.head()

## 4. Select Falcon 9 Launches

The project focuses on Falcon 9 launches. Rocket information is retrieved from the SpaceX API and used to identify Falcon 9 missions.

In [ ]:
rocket_response = requests.get("https://api.spacexdata.com/v4/rockets")
rocket_response.raise_for_status()

rockets = rocket_response.json()
rocket_lookup = {rocket["id"]: rocket["name"] for rocket in rockets}

df["rocket_name"] = df["rocket"].map(rocket_lookup)

df = df[df["rocket_name"].str.contains("Falcon 9", na=False)].copy()

print("Falcon 9 dataset shape:", df.shape)

## 5. Select Relevant Columns

In [ ]:
columns = [
    "name",
    "date_utc",
    "success",
    "rocket",
    "rocket_name",
    "launchpad",
    "payloads",
    "orbit",
    "cores"
]

columns = [column for column in columns if column in df.columns]
df = df[columns].copy()

df.head()

## 6. Handle Missing Values

In [ ]:
missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)

## 7. Convert Date Information

In [ ]:
df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce")

df["year"] = df["date_utc"].dt.year

df[["name", "date_utc", "year"]].head()

## 8. Create Launch Success Indicator

The `success` field is converted into a numeric landing/launch outcome indicator for later analysis.

In [ ]:
df["success"] = df["success"].astype("Int64")

print(df["success"].value_counts(dropna=False))

## 9. Remove Completely Empty Records

In [ ]:
df = df.dropna(how="all").reset_index(drop=True)

print("Final working dataset shape:", df.shape)
df.head()

## 10. Save the Prepared Dataset

In [ ]:
df.to_csv("spacex_dataset.csv", index=False)

print("Prepared SpaceX dataset saved successfully.")

## Conclusion

The SpaceX Falcon 9 launch data has been collected, inspected, cleaned, transformed, and prepared for exploratory analysis and predictive modeling.